In [62]:
# Movie Data Preprocessing
# Input:  data/movies_scraped.pkl
# Output: data/movies.pkl, data/similarity.pkl

import pandas as pd
import pickle
import string
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
except:
    import nltk
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))

stemmer = SnowballStemmer('english')
print('Imports ready')

Imports ready


## 1. Metadata

In [63]:
# Load scraped movie data
df = pd.read_pickle('data/movies_scraped.pkl')
print(f"Loaded {len(df)} movies")
df.head()

Loaded 13753 movies


,id,imdb_id,title,year,vote_average,vote_count,popularity,runtime,overview,tagline,genres,actors,directors,keywords,poster_url,watch_providers
0,2056,tt0340377,The Station Agent,2003.0,7.353,742,8.6303,88,"When his only friend dies, a man born with dwa...",Loneliness is much better when you have got so...,"[Drama, Comedy]","[Peter Dinklage, Patricia Clarkson, Bobby Cann...",[Tom McCarthy],"[friendship, new jersey, small person]",NaN,NaN
1,9659,tt0079501,Mad Max,1979.0,6.677,4772,8.7163,91,"In the ravaged near-future, a savage motorcycl...",The maximum force of the future.,"[Adventure, Action, Thriller, Science Fiction]","[Mel Gibson, Joanne Samuel, Hugh Keays-Byrne, ...",[George Miller],"[australia, petrol, baby, chain, dystopia, pos...",NaN,NaN
2,534075,tt7485048,Super 30,2019.0,7.211,121,2.2359,153,Based on life of Patna-based mathematician Ana...,,[Drama],"[Hrithik Roshan, Mrunal Thakur, Veerendra Saxe...",[Vikas Bahl],"[mathematician, biography, patna, bihar]",NaN,NaN
3,159092,tt2318527,Hell Baby,2013.0,5.119,236,5.4126,98,After she and her husband move into a haunted ...,The Devil got a baby mama.,"[Comedy, Horror]","[Rob Corddry, Leslie Bibb, Keegan-Michael Key,...","[Thomas Lennon, Robert Ben Garant]","[vatican (holy see), new orleans, louisiana, e...",NaN,NaN
4,710356,tt5616176,2 Hearts,2020.0,7.268,622,7.9719,102,When illness strikes two people who are polar ...,Discover the mystery that connects them all.,"[Romance, Drama]","[Jacob Elordi, Adan Canto, Tiera Skovbye, Radh...",[Lance Hool],"[faith, disease, organ transplant, destiny]",NaN,NaN


In [64]:
# Ensure list columns are proper lists
for col in ['genres', 'actors', 'directors', 'keywords']:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: x if isinstance(x, list) else [])
df['overview'] = df['overview'].fillna('')
df['tagline'] = df['tagline'].fillna('')
print('Data cleaned')

Data cleaned


In [65]:
# Process text and names
def process_text(text):
    if not text or not isinstance(text, str): return []
    text = text.lower()
    translator = str.maketrans('', '', string.punctuation)
    words = text.translate(translator).split()
    return [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]

def process_names(names):
    if not isinstance(names, list): return []
    return [name.lower().replace(' ', '') for name in names if name]

df['overview_processed'] = df['overview'].apply(process_text)
df['tagline_processed'] = df['tagline'].apply(process_text)
df['actors_processed'] = df['actors'].apply(process_names)
df['directors_processed'] = df['directors'].apply(process_names)
df['genres_processed'] = df['genres'].apply(lambda x: [g.lower() for g in x] if isinstance(x, list) else [])
df['keywords_processed'] = df['keywords'].apply(lambda x: [stemmer.stem(k.lower().replace(' ', '')) for k in x] if isinstance(x, list) else [])
print('Features processed')

Features processed


In [ ]:
# Create soup (combined features)
def create_soup(row):
    parts = []
    # Rating: Granular half-point buckets, high weight (quality matters)
    if pd.notna(row.get('vote_average')):
        rating = row['vote_average']
        bucket = int(rating * 2) / 2  # Rounds to nearest 0.5 (e.g., 8.3 -> 8.5, 8.7 -> 8.5)
        parts.extend([f"rating{bucket}"] * 3)  # High weight for quality
    
    # Genres: Important for basic categorization, medium weight
    parts.extend(row.get('genres_processed', []) * 2)
    
    # Overview: MOST IMPORTANT - this is the plot/story, highest weight
    parts.extend(row.get('overview_processed', []) * 3)  # Increased from 1
    
    # Keywords: Important for themes/concepts, high weight
    parts.extend(row.get('keywords_processed', []) * 2)  # Increased from 1
    
    # Tagline: Additional plot context, medium weight
    parts.extend(row.get('tagline_processed', []) * 1)  # Keep at 1
    
    # Actors/Directors: Lower weight (reduced from 2x to 1x)
    parts.extend(row.get('actors_processed', []) * 1)  # Reduced from 2
    parts.extend(row.get('directors_processed', []) * 1)  # Reduced from 2
    
    return ' '.join(parts)

df['soup'] = df.apply(create_soup, axis=1)
print('Soup created')

Soup created


In [67]:
# TF-IDF
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=3, max_df=0.8, max_features=20000, stop_words='english', dtype='float32')
print('Fitting TF-IDF...')
tfidf_matrix = tf.fit_transform(df['soup'].values.astype(str))
print(f'TF-IDF shape: {tfidf_matrix.shape}')

Fitting TF-IDF...


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


TF-IDF shape: (13753, 20000)


In [68]:
# Cosine similarity
print('Computing similarity...')
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f'Similarity shape: {cosine_sim.shape}')

Computing similarity...
Similarity shape: (13753, 13753)


In [69]:
# Test recommendations
df = df.reset_index(drop=True)
indices = pd.Series(df.index, index=df['title'])

def get_recommendations(title, n=10):
    if title not in indices: return None
    idx = indices[title]
    if isinstance(idx, pd.Series): idx = idx.iloc[0]
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    return df[['title', 'year', 'vote_average']].iloc[[i[0] for i in sim_scores]]

get_recommendations('The Dark Knight', 10)

,title,year,vote_average
4574,The Dark Knight Rises,2012.0,7.790
5140,Batman Begins,2005.0,7.720
10018,Batman,1989.0,7.200
1748,"Batman: The Long Halloween, Part One",2021.0,7.458
5408,"Batman: The Long Halloween, Part Two",2021.0,7.402
10860,The Batman,2022.0,7.660
3689,Batman Forever,1995.0,5.448
1427,The Prestige,2006.0,8.206
9458,Batman: Mask of the Phantasm,1993.0,7.482
5750,Following,1999.0,7.129


In [70]:
# Save output
with open('data/movies.pkl', 'wb') as f:
    pickle.dump(df, f)
with open('data/similarity.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)
print(f'Done! Saved {len(df)} movies.')

Done! Saved 13753 movies.


In [ ]:
# End of notebook - cells below are legacy

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

### 2. Credits

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

## 3. Keywords

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

## 4. Merging dataframes on id

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass

In [ ]:
pass